[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/misda/blob/feat/refine/examples/benchmark.ipynb)


# MISDA - Maximal Independent Structural Dimensionality Analysis

This notebook serves as a comprehensive benchmark suite for the Maximal Independent Structural Dimensionality Analysis (MISDA) framework.
It systematically evaluates the algorithm's efficacy across a spectrum of synthetic benchmarks, ranging from canonical correlation patterns—including linear redundancies and latent manifolds—to complex Multi-Objective Problems (MOPs). The analysis verifies MISDA's capability to correctly identify intrinsic dimensionality and preserve the topological fidelity of the Pareto frontier.

In [ ]:
# Install MISDA 
!pip install --upgrade git+https://github.com/monacofj/misda.git@feat/refine

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import misda

# Reload for development iteration
import importlib
importlib.reload(misda)

print("MISDA imported successfully.")


## 1. Data Generators & Utilities

### General Helpers

In [ ]:
def _truth(name, latent_expected, structural_expected, blocks_expected, feature="", intuition="", graph_expected=""):
    return {
        "name": name,
        "latent_expected": int(latent_expected) if latent_expected != "" and latent_expected is not None else None,
        "structural_expected": int(structural_expected) if structural_expected != "" and structural_expected is not None else None,
        "blocks_expected": blocks_expected,
        "feature": feature,
        "intuition": intuition,
        "graph_expected": graph_expected,
    }

def _mk_block_names(start, size):
    # start is 1-based
    return [f"f{i}" for i in range(start, start + size)]

def _repeat_with_small_noise(base, rng, noise):
    # base: (N,) -> returns perturbed (N,)
    return base + noise * rng.normal(size=base.shape[0])

def _mop_df(Y):
    return pd.DataFrame(Y, columns=[f"f{i+1}" for i in range(Y.shape[1])])


### Validation Utilities
Custom function to evaluate reconstruction fidelity on benchmark datasets.

### Run cases

In [ ]:
def run_cases(cases_list, N=300):
    results = {}
    for name, gen in cases_list:
        Y, truth = gen(N=N)
        problem = truth.get("feature", truth.get("notes", ""))
        intuition = truth.get("intuition", "")
        graph_expected = truth.get("graph_expected", "")
        latent = truth.get("latent_expected", truth.get("intrinsic_dim_expected", ""))
        structural = truth.get("structural_expected", truth.get("latent_expected", truth.get("intrinsic_dim_expected", "")))

        print(f"\n{'='*80}")
        print("Case description")
        print(f"Running:    {name}")
        if problem:
            print(f"Problem:    {problem}")
        if intuition:
            print(f"Intuition:  {intuition}")
        if latent != "" and latent is not None:
            print(f"Latent:     {latent}")
        if structural != "" and structural is not None:
            print(f"Structural: {structural}")
        if graph_expected:
            print(f"Graph:      {graph_expected}")
        print(f"{'='*80}")

        # --- 1. Execute MISDA analysis (using defaults: )
        result = misda.analyze(Y, name=name)
        result.validate()
        
        # --- 2. Full Technical Report ---
        print(result.report())

        # --- 3. Plot Graph ---
        try:
            result.plot()
        except Exception as e:
            print(f"Plotting failed: {e}")
        
        results[name] = {
            "result_obj": result,
            "truth": truth
        }
    return results


### Canonical Structure Test Suite (qualitative calibration)

In [ ]:
# --- Battery 1: Standard Correlation Cases ---

def make_case1_independence(N=1000, M=20, seed=123):
    rng = np.random.default_rng(seed)
    Y = rng.normal(size=(N, M))
    cols = [f"f{i+1}" for i in range(M)]
    df = pd.DataFrame(Y, columns=cols)
    truth = _truth(
        name="Case 1 - Total independence",
        latent_expected=M,
        structural_expected=M,
        blocks_expected=[[c] for c in cols],
        feature="All 20 objectives are mutually independent i.i.d. Gaussian noise variables.",
        intuition="20 completely unrelated random sensors; knowing one tells you nothing about any other. MISDA should keep all 20.",
        graph_expected="20 isolated nodes (0 edges, 20 connected components)"
    )
    return df, truth

def make_case2_total_redundancy(N=1000, M=20, seed=123):
    rng = np.random.default_rng(seed)
    latent = rng.normal(size=(N, 1))
    noise = rng.normal(scale=0.05, size=(N, M))
    Y = latent + noise
    cols = [f"f{i+1}" for i in range(M)]
    df = pd.DataFrame(Y, columns=cols)
    truth = _truth(
        name="Case 2 - Total redundancy",
        latent_expected=1,
        structural_expected=1,
        blocks_expected=[cols],
        feature="All 20 objectives are noisy linear copies of a single 1D latent factor.",
        intuition="20 identical thermometers measuring the exact same room temperature with minor noise. MISDA should keep just 1.",
        graph_expected="1 fully connected graph (K_20, 190 edges, 1 connected component)"
    )
    return df, truth

def make_case3_block_structure(N=1000, M=20, seed=123):
    rng = np.random.default_rng(seed)
    assert M == 20
    latent_blocks = rng.normal(size=(N, 4))
    Y = np.zeros((N, M))
    for b in range(4):
        for j in range(5):
            idx = 5*b + j
            Y[:, idx] = latent_blocks[:, b] + rng.normal(scale=0.2, size=N)
    cols = [f"f{i+1}" for i in range(M)]
    df = pd.DataFrame(Y, columns=cols)
    blocks = [
        [f"f{i}" for i in range(1, 6)],
        [f"f{i}" for i in range(6, 11)],
        [f"f{i}" for i in range(11, 16)],
        [f"f{i}" for i in range(16, 21)],
    ]
    truth = _truth(
        name="Case 3 - Blocks (4 x 5)",
        latent_expected=4,
        structural_expected=4,
        blocks_expected=blocks,
        feature="4 independent latent factors; each factor generates a cluster of 5 redundant objectives.",
        intuition="4 physical properties (e.g., Temp, Pressure, Humidity, Speed), each measured by 5 duplicate sensors. MISDA should reduce 20 sensors to 4.",
        graph_expected="4 disjoint complete subgraphs of 5 nodes each (4 x K_5, 40 total edges)"
    )
    return df, truth

def make_case4_two_big_blocks(N=1000, M=20, seed=123):
    rng = np.random.default_rng(seed)
    assert M == 20
    latent_blocks = rng.normal(size=(N, 2))
    Y = np.zeros((N, M))
    for i in range(10):
        Y[:, i] = latent_blocks[:, 0] + rng.normal(scale=0.2, size=N)
    for i in range(10, 20):
        Y[:, i] = latent_blocks[:, 1] + rng.normal(scale=0.2, size=N)
    cols = [f"f{i+1}" for i in range(M)]
    df = pd.DataFrame(Y, columns=cols)
    truth = _truth(
        name="Case 4 - Blocks (2 x 10)",
        latent_expected=2,
        structural_expected=2,
        blocks_expected=[
            [f"f{i}" for i in range(1, 11)],
            [f"f{i}" for i in range(11, 21)],
        ],
        feature="2 independent latent factors; each factor generates a cluster of 10 redundant objectives.",
        intuition="Measuring 2 goals (e.g., Cost and Weight), but using 10 duplicate formulas for Cost and 10 for Weight. MISDA should reduce 20 formulas to 2.",
        graph_expected="2 disjoint complete subgraphs of 10 nodes each (2 x K_10, 90 total edges)"
    )
    return df, truth

def make_case5_chain_structure(N=1000, M=20, seed=123):
    rng = np.random.default_rng(seed)
    Y = np.zeros((N, M))
    Y[:, 0] = rng.normal(size=N)
    for j in range(1, M):
        Y[:, j] = Y[:, j-1] + rng.normal(scale=0.2, size=N)
    cols = [f"f{i+1}" for i in range(M)]
    df = pd.DataFrame(Y, columns=cols)
    truth = _truth(
        name="Case 5 - Chain",
        latent_expected=M,
        structural_expected=M,
        blocks_expected=[cols],
        feature="Markovian random walk chain where correlation decays smoothly with index distance.",
        intuition="A chain of 20 dominoes: adjacent dominoes are strongly linked, but the 1st and 20th are far apart. Tests gradual, step-by-step continuous dependency.",
        graph_expected="1 connected chain/band graph (adjacent node edges, 1 connected component)"
    )
    return df, truth

def make_case6_mixed_structure(N=1000, M=20, seed=123):
    rng = np.random.default_rng(seed)
    assert M == 20
    Y = np.zeros((N, M))
    # First 10: independent
    Y[:, :10] = rng.normal(size=(N, 10))
    # Last 10: two latents
    latent1 = rng.normal(size=N)
    latent2 = rng.normal(size=N)
    for j in range(10, 15):
        Y[:, j] = latent1 + rng.normal(scale=0.2, size=N)
    for j in range(15, 20):
        Y[:, j] = latent2 + rng.normal(scale=0.2, size=N)
    cols = [f"f{i+1}" for i in range(M)]
    df = pd.DataFrame(Y, columns=cols)
    truth = _truth(
        name="Case 6 - Mixed (indep + latents)",
        latent_expected=12,
        structural_expected=12,
        blocks_expected=[[f"f{i}"] for i in range(1, 11)] + [[f"f{i}" for i in range(11, 16)], [f"f{i}" for i in range(16, 21)]],
        feature="Heterogeneous structure: 10 independent noise objectives (f1..f10) and 2 redundant blocks of 5.",
        intuition="10 random independent variables mixed with 2 redundant groups of 5 sensors each. MISDA should keep 10 + 2 = 12 objectives.",
        graph_expected="10 isolated nodes and 2 disjoint complete subgraphs of 5 nodes each (10 x K_1 + 2 x K_5)"
    )
    return df, truth

def make_case7_pure_conflict_groups(N=1000, M=20, noise=0.05, seed=123, **kwargs):
    rng = np.random.default_rng(seed)
    if M < 2:
        raise ValueError("M must be >= 2")
    M_pos = (M + 1) // 2
    M_neg = M - M_pos
    x = rng.normal(size=N)
    Y_pos = np.column_stack([x + noise * rng.normal(size=N) for _ in range(M_pos)])
    Y_neg = np.column_stack([(-x) + noise * rng.normal(size=N) for _ in range(M_neg)])
    Y = np.column_stack([Y_pos, Y_neg])
    cols = [f"f{i+1}" for i in range(M)]
    Y = pd.DataFrame(Y, columns=cols)
    truth = _truth(
        name="Case 7 - Structural conflict (anti-corr) 2-groups",
        latent_expected=1,
        structural_expected=2,
        blocks_expected=[cols[:M_pos], cols[M_pos:]],
        feature="Two groups (+x and -x) with internal redundancy and strong structural conflict (anti-correlation).",
        intuition="10 sensors measuring Car Speed (+x) and 10 measuring Remaining Travel Time (-x). Speed and Time conflict, but both are essential! MISDA must keep 1 of each.",
        graph_expected="2 disjoint complete subgraphs of 10 nodes each (2 x K_10, 90 total edges, 2 connected components)"
    )
    return Y, truth


### Synthetic MOP Test Suite (nferential validation)

In [ ]:
# --- Battery 2: MOP Benchmarks ---

def mopA_monotonic_redundancy(N=1000, seed=123, noise=0.0):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0.0, 1.0, size=N)

    # 20 monotonic transformations (all 1D redundant)
    feats = [
        x,
        2.0 * x + 0.1,
        np.log(1.0 + 9.0 * x),
        x**2,
        np.sqrt(np.maximum(x, 0.0)),
        x**3,
        np.exp(0.5 * x) - 1.0,
        1.0 / (1.0 + np.exp(-10.0 * (x - 0.5))),
        (x + 0.2) ** 2,
        np.log(1.0 + 3.0 * x),
        np.tanh(2.0 * x),
        (1.0 + x) ** 1.5,
        np.clip(x + 0.05, 0, 1),
        np.clip(1.2 * x, 0, 1),
        np.log1p(20.0 * x) / np.log1p(20.0),
        (x + 1e-6) ** 0.25,
        (x + 0.1) ** 3,
        np.sqrt(np.maximum(0.1 + x, 0.0)),
        np.exp(x) - 1.0,
        (x + 0.3) ** 2,
    ]
    Y = np.vstack([_repeat_with_small_noise(f, rng, noise) for f in feats]).T

    truth = _truth(
        name="MOP-A — Monotonic redundancy (1D, M=20)",
        latent_expected=1,
        structural_expected=1,
        blocks_expected=[_mk_block_names(1, 20)],
        feature="20 non-linear monotonic transformations driven by a single 1D decision variable x.",
        intuition="20 different formulas (squares, roots, logs) calculated from a single input x. Since all move in sync, MISDA should collapse all 20 to 1.",
        graph_expected="1 fully connected graph (K_20, 190 edges, 1 connected component)"
    )
    return _mop_df(Y), truth

def mopB_tradeoff_with_redundancies(N=1000, seed=123, noise=0.02):
    rng = np.random.default_rng(seed)
    a = rng.uniform(0.0, 1.0, size=N)
    b = rng.uniform(0.0, 1.0, size=N)

    # Plausible latents
    C = 0.6 * a + 0.8 * b            # cost in ~[0,1.4]
    E = b + 0.3 * (1.0 - a)          # consumption in ~[0,1.3]
    P = a * (1.0 - b) + 0.2 * a      # performance can go up to 1.2 -> BUG for Q

    # FIX: force performance to stay in [0,1] so that Q=1-P stays in [0,1]
    P = np.clip(P, 0.0, 1.0)
    Q = 1.0 - P

    # 7 "cost" objectives
    cost_feats = [
        C,
        _repeat_with_small_noise(C, rng, noise),
        1.0 + 2.0 * C,
        np.log1p(9.0 * C),
        np.sqrt(np.maximum(C, 0.0)),
        C**2,
        (C + 0.1) ** 1.5,
    ]

    # 7 "consumption" objectives
    cons_feats = [
        E,
        _repeat_with_small_noise(E, rng, noise),
        np.sqrt(np.maximum(E, 0.0)),
        np.log1p(9.0 * E),
        E**2,
        (E + 0.05),
        (E + 0.2) ** 1.3,
    ]

    # 6 "performance" objectives (minimization via 1-P), with protected domain
    Q_rep = np.clip(_repeat_with_small_noise(Q, rng, noise), 0.0, 1.0)

    perf_feats = [
        Q,
        Q_rep,
        Q**2,
        np.sqrt(np.maximum(Q, 0.0)),
        np.log1p(9.0 * Q),            # now Q ∈ [0,1] -> always valid
        (Q + 0.1) ** 1.2,             # now Q+0.1 ∈ [0.1,1.1] -> always valid
    ]

    feats = cost_feats + cons_feats + perf_feats
    Y = np.vstack(feats).T

    truth = _truth(
        name="MOP-B — Trade-off + redundancies (~2D, M=20)",
        latent_expected=2,
        structural_expected=2,
        blocks_expected=[_mk_block_names(1, 7), _mk_block_names(8, 7), _mk_block_names(15, 6)],
        feature="Three functional engineering families (7 cost, 7 consumption, 6 performance) driven by 2 decision variables.",
        intuition="An engineering problem with 3 main goals: Cost, Energy, and Performance, each measured in multiple redundant ways. MISDA should shrink 20 to ~2-3 core trade-offs.",
        graph_expected="1 connected graph with 3 dense functional clusters (1 connected component, effective dim ~2)"
    )
    return _mop_df(Y), truth

def mopC_latent_blocks_4x5(N=1000, seed=123, noise=0.02):
    rng = np.random.default_rng(seed)
    u, v, w, z = rng.uniform(0.0, 1.0, size=(4, N))
    eps = rng.normal(size=N)

    b1 = [u, 2*u, u**2, np.sqrt(np.maximum(u,0.0)), np.log1p(9*u)]
    b2 = [v, v+0.5, np.log1p(9*v), v**2, np.sqrt(np.maximum(v,0.0))]
    b3 = [w, w+noise*eps, np.sqrt(np.maximum(w,0.0)), np.log1p(9*w), (w+0.1)**2]
    b4 = [z, (1.0+z)**2, np.exp(z)-1.0, np.log1p(9*z), np.sqrt(np.maximum(z,0.0))]

    feats = b1 + b2 + b3 + b4
    Y = np.vstack(feats).T

    truth = _truth(
        name="MOP-C — Latent blocks (4×5, M=20)",
        latent_expected=4,
        structural_expected=4,
        blocks_expected=[_mk_block_names(1,5), _mk_block_names(6,5), _mk_block_names(11,5), _mk_block_names(16,5)],
        feature="4 independent decision factors; each factor generates a block of 5 non-linearly transformed objectives.",
        intuition="4 control dials, where turning each dial affects 5 non-linear indicators. MISDA should extract 4 independent representatives (1 per dial).",
        graph_expected="4 disjoint dense subgraphs of 5 nodes each (4 x K_5, 4 connected components)"
    )
    return _mop_df(Y), truth

def mopD_pure_conflict_groups(N=1000, seed=123, noise=0.0):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0.0, 1.0, size=N)

    g1 = [
        x,
        2*x + 0.1,
        np.log1p(9*x),
        x**2,
        np.sqrt(np.maximum(x,0.0)),
        x**3,
        np.tanh(2*x),
        np.log1p(3*x),
        (x+0.2)**2,
        (1.0 + x)**1.5,
    ]
    y = 1.0 - x
    g2 = [
        y,
        2*y + 0.1,
        np.log1p(9*y),
        y**2,
        np.sqrt(np.maximum(y,0.0)),
        y**3,
        np.tanh(2*y),
        np.log1p(3*y),
        (y+0.2)**2,
        (1.0 + y)**1.5,
    ]

    feats = [_repeat_with_small_noise(f, rng, noise) for f in (g1 + g2)]
    Y = np.vstack(feats).T

    truth = _truth(
        name="MOP-D — Structural conflict (anti-corr) 2-groups (M=20)",
        latent_expected=1,
        structural_expected=2,
        blocks_expected=[_mk_block_names(1,10), _mk_block_names(11,10)],
        feature="Two antagonistic non-linear objective families (+x vs 1-x) with internal redundancy and trade-off conflict.",
        intuition="10 indicators measuring Benefit (+x) vs 10 measuring Risk (1-x). Benefit and Risk directly conflict. MISDA must preserve 1 Benefit and 1 Risk indicator.",
        graph_expected="2 disjoint complete subgraphs of 10 nodes each (2 x K_10, 90 total edges, 2 connected components)"
    )
    return _mop_df(Y), truth

def mopE_partial_redundancy_noisy(N=1000, seed=123, noise=0.05):
    rng = np.random.default_rng(seed)
    a = rng.uniform(0.0, 1.0, size=N)
    b = rng.uniform(0.0, 1.0, size=N)
    eps = rng.normal(size=N)

    # subfamily A: redundant around 'a' (10)
    A = [
        a,
        a + noise*eps,
        a - noise*eps,
        2*a + 0.1,
        a**2,
        np.sqrt(np.maximum(a,0.0)),
        np.log1p(9*a),
        (a+0.2)**2,
        np.tanh(2*a),
        (1.0+a)**1.2,
    ]

    # subfamily B: "b" (4)
    B = [
        b,
        b + 0.5,
        np.sqrt(np.maximum(b,0.0)),
        np.log1p(9*b),
    ]

    # mixtures/compounds: functions of s=a+b (6)
    s = a + b
    C = [
        s,
        s**2,
        np.sqrt(np.maximum(s,0.0)),
        np.log1p(9*s),
        (s+0.1)**1.5,
        1.0/(1.0+np.exp(-10*(s-1.0))),
    ]

    feats = A + B + C
    Y = np.vstack(feats).T

    truth = _truth(
        name="MOP-E — Partial redundancy + noise (M=20)",
        latent_expected=2,
        structural_expected=2,
        blocks_expected=[_mk_block_names(1,10), _mk_block_names(11,4), _mk_block_names(15,6)],
        feature="Partial redundancy across 2 latent drivers (a,b): 10 objectives on a, 4 on b, and 6 compounds on s=a+b.",
        intuition="Overlapping signals: some indicators monitor Engine A, some monitor Engine B, and some monitor both combined (A+B). Tests if MISDA untangles blended signals.",
        graph_expected="1 connected graph with 3 dense overlapping clusters (Subfamily A: 10, B: 4, C: 6)"
    )
    return _mop_df(Y), truth

def mopF_regime_switching(N=1000, seed=123, sharpness=20.0, noise=0.0):
    rng = np.random.default_rng(seed)
    a = rng.uniform(0.0, 1.0, size=N)
    b = rng.uniform(0.0, 1.0, size=N)

    s = 1.0 / (1.0 + np.exp(-sharpness * (a - 0.5)))
    L = (1.0 - s) * a + s * b

    eps = rng.normal(size=N)

    L_feats = [
        L,
        L**2,
        np.log1p(9*L),
        np.sqrt(np.maximum(L,0.0)),
        (L+0.1)**1.5,
        np.tanh(2*L),
        np.exp(0.5*L)-1.0,
        (L+0.2)**2,
        np.log1p(3*L),
        _repeat_with_small_noise(L, rng, 0.02) if noise == 0.0 else _repeat_with_small_noise(L, rng, noise),
    ]

    b_feats = [
        b,
        np.sqrt(np.maximum(b,0.0)),
        np.log1p(9*b),
        b**2,
        (b+0.1)**1.5,
        np.tanh(2*b),
        np.exp(0.5*b)-1.0,
        (b+0.2)**2,
        np.log1p(3*b),
        _repeat_with_small_noise(b, rng, 0.02) if noise == 0.0 else _repeat_with_small_noise(b, rng, noise),
    ]

    feats = L_feats + b_feats
    Y = np.vstack(feats).T

    truth = _truth(
        name="MOP-F — Regimes (mixture, M=20)",
        latent_expected=2,
        structural_expected=2,
        blocks_expected=[_mk_block_names(1,10), _mk_block_names(11,10)],
        feature="Non-linear regime-switching mixture: 10 objectives on regime-dependent mixture L(a,b) and 10 on b.",
        intuition="System switching: indicators change behavior depending on whether the system operates in High-Power or Low-Power mode. Tests MISDA under shifting states.",
        graph_expected="1 connected graph with 2 dense interconnected clusters (10 on mixture L, 10 on b)"
    )
    return _mop_df(Y), truth


## Run test batteries

In [ ]:
battery1 = [
    ("Case 1 - Total independence", make_case1_independence),
    ("Case 2 - Total redundancy", make_case2_total_redundancy),
    ("Case 3 - Blocks (4 x 5)", make_case3_block_structure),
    ("Case 4 - Blocks (2 x 10)", make_case4_two_big_blocks),
    ("Case 5 - Chain", make_case5_chain_structure),
    ("Case 6 - Mixed (indep + latents)", make_case6_mixed_structure),
    ("Case 7 - Structural conflict (anti-corr) with groups", make_case7_pure_conflict_groups),
]

print("\n=== RUNNING STANDARD CORRELATION BATTERY ===")
battery1_results = run_cases(battery1)

battery1_fidelity_df = misda.compile_benchmark_summary(battery1_results)
print("\n--- MISDA Reconstruction Fidelity Evaluation for Canonical Cases ---")
print(battery1_fidelity_df.to_markdown(index=False))


In [ ]:
battery2 = [
    ("MOP-A — Monotonic redundancy", mopA_monotonic_redundancy),
    ("MOP-B — Trade-off + redundancies", mopB_tradeoff_with_redundancies),
    ("MOP-C — Latent blocks", mopC_latent_blocks_4x5),
    ("MOP-D — Pure conflict groups", mopD_pure_conflict_groups),
    ("MOP-E — Partial redundancy + noise", mopE_partial_redundancy_noisy),
    ("MOP-F — Regime switching", mopF_regime_switching),
]

print("\n\n=== RUNNING MOP BENCHMARK BATTERY ===")
mop_results = run_cases(battery2)

mop_fidelity_df = misda.compile_benchmark_summary(mop_results)
print("\n--- MISDA Reconstruction Fidelity Evaluation for MOP Cases ---")
print(mop_fidelity_df.to_markdown(index=False))


# 3. Conclusions

The benchmarking suite evaluates the behavior of the MISDA algorithm across a diverse set of structural topologies, yielding the following observations:

### Resolving the Signal from the Noise
In the canonical test cases, the algorithm distinguishes between varying types of dependence. It removed full linear redundancies (Case 2) while maintaining the dimensions of independent variables (Case 1). In more complex scenarios, such as the "Chain" structure (Case 5) or "Mixed Structure" (Case 6), the algorithm identified the underlying latent drivers by analyzing the connectivity of the partial correlation graph, rather than solely pairwise relationships. Limitations were noted in Case 4 (Transitive), where the algorithm relies on the chosen alpha threshold to distinguish indirect from direct correlations.

### The Pragmatism of MISDA in MOPs
When applied to synthetic Multi-Objective Problems (MOPs), which mimic nonlinear engineering landscapes, the results highlighted specific behaviors:
*   **MOP-A (Monotonic Redundancy):** The algorithm reduced the 20 transformed objectives to the single underlying variable, confirming robustness to monotonic transformations.
*   **MOP-D (Conflict Groups):** By identifying independent sets, the algorithm preserved the representative objectives from conflicting groups, retaining the dimensional structure of the trade-offs.
*   **MOP-B & C (Non-Linear):** The method showed sensitivity to highly non-linear relationships, where the linear correlation assumption ($r$) may underestimate dependence.
*   **MOP-E & F (Regime Switching):** In cases of low signal-to-noise ratio, the "Caution" parameter and the adaptive alpha threshold determined whether the algorithm defaulted to a conservative (keep all) or aggressive (prune) strategy.

Ultimately, these benchmarks indicate that MISDA functions as a **structural inference engine**. It separates essential conflict from redundant noise, providing a minimal representation of the decision space that attempts to preserve the integrity of the Pareto frontier.